# STAC AI Workshop · NYC · 21 October 2026
## Section 4 · Diagnose the failure before changing the model
**Participant workbook · Matthew Dixon, PhD, FRM**

**Question:** Does a bad action come from the evidence, figure selection, calculation, interpretation or action rule?

This notebook executes a **controlled synthetic workflow**, with deliberately injected faults. Its traces are observable program steps, not reconstructed LLM thoughts. No GPU, API, training or future return is used. The separate recorded LoRA notebook contains the actual measured model comparison.

**Objectives:** trace the first failed observable check; distinguish potentially affected paths from actual changes; apply a verified replacement and rerun; choose a repair before deciding whether adaptation is justified.


## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/04_adaptation_lab.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/04_adaptation_lab.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:4** | `root=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'workflow_diagnosis.py').exists()),None)` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [1]:
# C01 · code cell
from pathlib import Path
import sys, json

import stac_ai_bootcamp.workflow_diagnosis as engine
import stac_ai_bootcamp.adaptation_display as view

In [2]:
experiment = engine.build()
view.panel(
    "Executed exercise · ready",
    view.table(
        ["Verification", "Result"],
        [[k, v] for k, v in experiment["verification"].items()],
    ),
    experiment["scope"],
)

Verification,Result
assertions_passed,59
controlled_cases,5
intervention_replays,30
negative_control,"Changing only the action cannot resolve source, extraction or tool-argument failures."
independent_branch,authority unchanged in every intervention replay


### 1 · Select and inspect a failure
Start with `CASE_ID='period'`. Then compare `basis`, `arguments`, `action`, and `two_faults`.

The execution gate may withhold several cases for different reasons. A shared final action does not imply a shared failure.


<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:4** | `before=engine.evaluate(case)` | Check the observable stages of this supplied synthetic workflow. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [3]:
# C02 · code cell
CASE_ID = "period"
case = next(c for c in engine.CASES if c["id"] == CASE_ID)
before = engine.evaluate(case)
view.panel(
    case["title"],
    view.table(
        ["Stage", "Observed value", "Check"],
        [
            [
                n["label"],
                before["nodes"][n["id"]]["value"],
                before["nodes"][n["id"]]["check"],
            ]
            for n in engine.GRAPH
        ],
    ),
    "First observable failed check: " + str(before["first_observable_failure"]),
)
view.panel(
    "Declared evidence and policy",
    view.details("Supplied reference document", experiment["reference_document"])
    + view.bullets(
        [
            "Require Harbor Metals, 2026 Q2, reported EPS.",
            "Surprise = (reported EPS − consensus EPS) / |consensus EPS|.",
            "Signal is POSITIVE only above 10%; BUY for POSITIVE, otherwise HOLD.",
            "The gate withholds any proposal when a required observable check fails.",
        ]
    ),
    "These rules and reference fields are supplied for the exercise. They are not automatically discovered financial truth.",
)

Stage,Observed value,Check
Retrieved document,"{ ""issuer"": ""Harbor Metals"", ""period"": ""2026 Q1"", ""reported_eps"": 0.8, ""forecast_eps"": 0.8, ""consensus_eps"": 1.0, ""published_at"": ""2026-07-01"", ""id"": ""HM-Q1-release"" }",FAIL
Selected reported EPS,"{ ""value"": 0.8, ""basis"": ""reported"", ""source_id"": ""HM-Q1-release"" }",PASS
Calculator arguments,"{ ""actual"": 0.8, ""consensus"": 1.0 }",PASS
Calculated surprise,-0.2,PASS
Signal interpretation,NON_POSITIVE,PASS
Proposed action,HOLD,PASS
Reported earnings summary,Reported EPS: 0.80,FAIL
Negative-surprise flag,True,FAIL
Delegated order permission,True,PASS
Execution gate,WITHHOLD,FAIL


<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:2** | `view.panel('Dependency graph · inspect the branches',view.table(['Node','Type','Depends on'],[[n['id'],n['type'],', '.join(n['parents']) or 'Independent root'] for n in engine.GRAPH]),'The selected figure also feeds the earnings report. The calculation also feeds a negative-surprise flag. The delegated permission is an independent branch.')` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [4]:
# C03 · code cell
view.panel(
    "Dependency graph · inspect the branches",
    view.table(
        ["Node", "Type", "Depends on"],
        [
            [n["id"], n["type"], ", ".join(n["parents"]) or "Independent root"]
            for n in engine.GRAPH
        ],
    ),
    "The selected figure also feeds the earnings report. The calculation also feeds a negative-surprise flag. The delegated permission is an independent branch.",
)

Node,Type,Depends on
source,evidence,Independent root
selected,extraction,source
tool_request,tool_request,selected
tool_result,tool_result,tool_request
interpretation,claim,tool_result
recommendation,action,interpretation
report,claim,selected
risk,claim,tool_result
authority,permission,Independent root
execution,gate,"source, selected, tool_request, tool_result, interpretation, recommendation, authority"


### 2 · Predict what a repair will change
Choose `source`, `selected`, `tool_request`, `recommendation` or `all`.

Before running, write down which checks should pass, which dependent outputs might change, and whether another fault could remain. A replacement fixes only its selected stage; downstream faulty steps are retained.


<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:5** | `result=engine.run(case,intervention)` | Apply the selected controlled replacement and recompute dependent checks. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [5]:
# C04 · code cell
REPAIR_ID = "source"
my_prediction = "Write your prediction before applying the replacement."
intervention = next(x for x in engine.INTERVENTIONS if x["id"] == REPAIR_ID)
result = engine.run(case, intervention)
after = result["after"]
view.panel(
    "Controlled intervention · " + intervention["label"],
    view.table(
        ["Quantity", "Computed result"],
        [
            ["Potential descendants", result["potential_descendants"]],
            ["Actually changed nodes", result["actually_changed_nodes"]],
            ["Remaining failed checks", result["remaining_failed_checks"]],
            [
                "Proposed action before / after",
                before["proposed_action"] + " / " + after["proposed_action"],
            ],
            [
                "Permitted action before / after",
                before["terminal_action"] + " / " + after["terminal_action"],
            ],
        ],
    ),
    "Replacements are supplied verified exercise values. This is conditional stage localization, not proof of a hidden neural mechanism.",
)
view.panel(
    "Before and after",
    view.table(
        ["Node", "Before value / check", "After value / check"],
        [
            [
                n["id"],
                {
                    "value": before["nodes"][n["id"]]["value"],
                    "check": before["nodes"][n["id"]]["check"],
                },
                {
                    "value": after["nodes"][n["id"]]["value"],
                    "check": after["nodes"][n["id"]]["check"],
                },
            ]
            for n in engine.GRAPH
        ],
    ),
)

Quantity,Computed result
Potential descendants,"[ ""selected"", ""tool_request"", ""tool_result"", ""interpretation"", ""recommendation"", ""report"", ""risk"", ""execution"" ]"
Actually changed nodes,"[ ""source"", ""selected"", ""tool_request"", ""tool_result"", ""interpretation"", ""recommendation"", ""report"", ""risk"", ""execution"" ]"
Remaining failed checks,[]
Proposed action before / after,HOLD / BUY
Permitted action before / after,WITHHOLD / BUY


Node,Before value / check,After value / check
source,"{ ""value"": { ""issuer"": ""Harbor Metals"", ""period"": ""2026 Q1"", ""reported_eps"": 0.8, ""forecast_eps"": 0.8, ""consensus_eps"": 1.0, ""published_at"": ""2026-07-01"", ""id"": ""HM-Q1-release"" }, ""check"": ""FAIL"" }","{ ""value"": { ""issuer"": ""Harbor Metals"", ""period"": ""2026 Q2"", ""reported_eps"": 1.2, ""forecast_eps"": 0.8, ""consensus_eps"": 1.0, ""published_at"": ""2026-07-01"", ""id"": ""HM-Q2-release"" }, ""check"": ""PASS"" }"
selected,"{ ""value"": { ""value"": 0.8, ""basis"": ""reported"", ""source_id"": ""HM-Q1-release"" }, ""check"": ""PASS"" }","{ ""value"": { ""value"": 1.2, ""basis"": ""reported"", ""source_id"": ""HM-Q2-release"" }, ""check"": ""PASS"" }"
tool_request,"{ ""value"": { ""actual"": 0.8, ""consensus"": 1.0 }, ""check"": ""PASS"" }","{ ""value"": { ""actual"": 1.2, ""consensus"": 1.0 }, ""check"": ""PASS"" }"
tool_result,"{ ""value"": -0.2, ""check"": ""PASS"" }","{ ""value"": 0.2, ""check"": ""PASS"" }"
interpretation,"{ ""value"": ""NON_POSITIVE"", ""check"": ""PASS"" }","{ ""value"": ""POSITIVE"", ""check"": ""PASS"" }"
recommendation,"{ ""value"": ""HOLD"", ""check"": ""PASS"" }","{ ""value"": ""BUY"", ""check"": ""PASS"" }"
report,"{ ""value"": ""Reported EPS: 0.80"", ""check"": ""FAIL"" }","{ ""value"": ""Reported EPS: 1.20"", ""check"": ""PASS"" }"
risk,"{ ""value"": true, ""check"": ""FAIL"" }","{ ""value"": false, ""check"": ""PASS"" }"
authority,"{ ""value"": true, ""check"": ""PASS"" }","{ ""value"": true, ""check"": ""PASS"" }"
execution,"{ ""value"": ""WITHHOLD"", ""check"": ""FAIL"" }","{ ""value"": ""BUY"", ""check"": ""PASS"" }"


### 3 · The upstream repair is not always sufficient
Run the double-fault case below. Correct extraction restores the report, but the incorrect calculator argument still blocks execution. Do not infer “train the model” from a final bad decision.


<!-- WORKSHOP-CODE-GUIDE C05 -->
### Code cell C05 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C05:3** | `partial=engine.run(double,next(i for i in engine.INTERVENTIONS if i['id']=='selected'))` | Apply the selected controlled replacement and recompute dependent checks. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [6]:
# C05 · code cell
double = next(c for c in engine.CASES if c["id"] == "two_faults")
partial = engine.run(
    double, next(i for i in engine.INTERVENTIONS if i["id"] == "selected")
)
complete = engine.run(double, next(i for i in engine.INTERVENTIONS if i["id"] == "all"))
assert partial["remaining_failed_checks"] == ["tool_request"]
assert complete["restored"]
view.panel(
    "One repair can expose the next failure",
    view.table(
        ["Intervention", "Remaining failed checks", "Execution"],
        [
            [
                "Correct extraction only",
                partial["remaining_failed_checks"],
                partial["after"]["terminal_action"],
            ],
            [
                "Repair all declared stages",
                complete["remaining_failed_checks"],
                complete["after"]["terminal_action"],
            ],
        ],
    ),
    "The graph makes the unresolved dependency visible. Training cannot fix a calculator call whose arguments are supplied incorrectly by application code.",
)

Intervention,Remaining failed checks,Execution
Correct extraction only,"[ ""tool_request"" ]",WITHHOLD
Repair all declared stages,[],BUY


### 4 · Choose the remedy, then qualify it
- Wrong period: repair retrieval filters and source eligibility.
- Wrong figure/basis: repair extraction or its checks; consider adaptation only after defining and testing that task.
- Wrong calculator inputs: repair tool argument construction and validation.
- Action violates the signal: enforce the declared action policy; a preference-trained model is not an enforcement boundary.

The independent permission branch remains unchanged in all 30 replay conditions. Changed descendants are computed from the graph; no causal claim is made about unobserved model internals. The five controlled cases are instructional evidence, not a production validation set.

**Exit task:** state the first observable failure, which replacement changed the outcome, what remained unresolved, and why your proposed repair is appropriate. Then open `04_actual_adaptation_readable.ipynb` to inspect the separate measured 16-case LoRA comparison.


## Answer checks · default configuration

- `period` first fails at **source**; `basis` at **selected**; `arguments` at **tool_request**; `action` at **recommendation**.
- The matching replacement repairs the single-fault case. An action-only change cannot repair an upstream source or extraction failure.
- In `two_faults`, replacing the selected figure leaves **tool_request** unresolved. Repairing all declared stages resolves the supplied workflow.
- Corrected single positive examples have surprise **+20%** and the declared action **BUY**. The negative action case has **−20%** and declared action **HOLD**. These are teaching rules, not demonstrated market alpha.
- Authority is an independent branch and stays unchanged in these interventions. Potential descendants show possible impact; actually changed nodes show observed recomputation.
- This is controlled graph-based workflow diagnosis. It does not retrain an LLM or establish a hidden neural cause.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.